In [1]:
import pandas as pd

In [2]:
daily_rainfall = "/home/dread/Documents/Personal projects/SA-Water-Dam-Level-Predictor/data/processed/daily_rainfall.csv"
clean_dam = "/home/dread/Documents/Personal projects/SA-Water-Dam-Level-Predictor/data/processed/clean_dam.csv"

In [3]:
clean_dam_df = pd.read_csv(clean_dam)
daily_rainfall_df = pd.read_csv(daily_rainfall)
print(daily_rainfall_df.info())
print(f"daily rainfall shape:{daily_rainfall_df.shape}")
print(clean_dam_df.info())
print(f"daily rainfall shape:{daily_rainfall_df.shape}")

<class 'pandas.DataFrame'>
RangeIndex: 9617 entries, 0 to 9616
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            9617 non-null   str    
 1   water_level_mm  9617 non-null   float64
 2   year            9617 non-null   int64  
 3   month           9617 non-null   int64  
dtypes: float64(1), int64(2), str(1)
memory usage: 300.7 KB
None
daily rainfall shape:(9617, 4)
<class 'pandas.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   willDam     186 non-null    str    
 1   River       186 non-null    str    
 2   Indicators  186 non-null    str    
 3   FSC         186 non-null    float64
 4   This Week   186 non-null    float64
 5   Last Week   186 non-null    str    
 6   Last Year   186 non-null    str    
dtypes: float64(2), str(5)
memory usage: 10.3 KB
None
daily rainfall shape:(9

In [4]:
###These are the rollling lag features being created: lag being 7, 14, 28, 56 and 84###
daily_rainfall_df["lag_7"] = daily_rainfall_df["water_level_mm"].shift(7)
daily_rainfall_df["lag_14"] = daily_rainfall_df["water_level_mm"].shift(14)
daily_rainfall_df["lag_28"] = daily_rainfall_df["water_level_mm"].shift(28)
daily_rainfall_df["lag_56"] = daily_rainfall_df["water_level_mm"].shift(56)
daily_rainfall_df["lag_84"] = daily_rainfall_df["water_level_mm"].shift(84)

print(daily_rainfall_df.info())
print(f"daily rainfall shape:{daily_rainfall_df.shape}")
print(f"lag_values\n:{daily_rainfall_df[["lag_7", "lag_14", "lag_84"]].tail(10)}")

<class 'pandas.DataFrame'>
RangeIndex: 9617 entries, 0 to 9616
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            9617 non-null   str    
 1   water_level_mm  9617 non-null   float64
 2   year            9617 non-null   int64  
 3   month           9617 non-null   int64  
 4   lag_7           9610 non-null   float64
 5   lag_14          9603 non-null   float64
 6   lag_28          9589 non-null   float64
 7   lag_56          9561 non-null   float64
 8   lag_84          9533 non-null   float64
dtypes: float64(6), int64(2), str(1)
memory usage: 676.3 KB
None
daily rainfall shape:(9617, 9)
lag_values
:         lag_7     lag_14    lag_84
9607  0.847092   0.311230  1.278165
9608  2.786286   2.848348  1.955726
9609  2.604120   0.321911  4.560972
9610  7.049541   1.401302  2.280130
9611  2.314826  12.050176  3.267598
9612  3.228934   1.274137  2.405660
9613  0.883835   1.328130  1.248567
9614  4.999110

In [5]:
"""This is the rolling window feaatures"""
daily_rainfall_df["rolling_mean_28"] = daily_rainfall_df["water_level_mm"].rolling(28).mean()
daily_rainfall_df["rolling_mean_84"] = daily_rainfall_df["water_level_mm"].rolling(84).mean()
daily_rainfall_df["weekly_sum"] = daily_rainfall_df["water_level_mm"].rolling(7).sum()
daily_rainfall_df["lag_7_weekly_sum"] = daily_rainfall_df["weekly_sum"].shift(7)
daily_rainfall_df["week_over_week_pct"] = (daily_rainfall_df["weekly_sum"]-daily_rainfall_df["lag_7_weekly_sum"])/daily_rainfall_df["lag_7_weekly_sum"] * 100
daily_rainfall_df["week_over_week_pct"] = daily_rainfall_df["week_over_week_pct"].fillna(0)

In [8]:
print(daily_rainfall_df["week_over_week_pct"])
print(daily_rainfall_df["water_level_mm"].describe())

0        0.000000
1        0.000000
2        0.000000
3        0.000000
4        0.000000
          ...    
9612    -2.077312
9613    -4.257066
9614   -41.844659
9615   -71.176527
9616   -64.622910
Name: week_over_week_pct, Length: 9617, dtype: float64
count    9617.000000
mean        1.149207
std         1.597756
min         0.000000
25%         0.080735
50%         0.448099
75%         1.633875
max        19.466718
Name: water_level_mm, dtype: float64


In [ ]:
count = 0
true_count_lst = []
percent_75 = 1.633875
daily_rainfall_df["significant_rain_days"] = daily_rainfall_df["water_level_mm"]>percent_75



0        True
1        True
2        True
3        True
4        True
        ...  
9612    False
9613    False
9614    False
9615    False
9616     True
Name: significant_rain_days, Length: 9617, dtype: bool


In [17]:
count = 0
true_count_lst = []
for values in daily_rainfall_df["significant_rain_days"]:
    if values == False:
        count += 1
    else:
        count = 0
    true_count_lst.append(count)
daily_rainfall_df["days_since_significant_rain"] = true_count_lst
print(daily_rainfall_df["days_since_significant_rain"].head(15))


0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     1
9     2
10    3
11    4
12    5
13    6
14    0
Name: days_since_significant_rain, dtype: int64
